# Operations and Storage

The **Operation** is Wintermute's central container. It composes all other domain objects:
analysts, devices, services, vulnerabilities, users, cloud accounts, test plans, and test runs.

This notebook walks through:
1. Creating an Operation and adding analysts/users
2. Modeling devices and services
3. Attaching vulnerability findings
4. Cloud account modeling (AWS)
5. Serialization with `to_dict()` / `from_dict()`
6. JSON file backend for persistence

In [ ]:
from wintermute.core import Operation

op = Operation("acme-pentest-2025")
op.addAnalyst("Jane Doe", "jdoe", "jane@acme.com")
op.addUser(
    uid="rsmith", name="Robert Smith", email="robert@acme.com", teams=["red-team"]
)

print(f"Operation: {op.operation_name}")
print(f"Analysts: {[(a.name, a.userid) for a in op.analysts]}")
print(f"Users: {[(u.uid, u.name) for u in op.users]}")

## Devices and Services

A **Device** represents a target host. Each device can hold **Services** (network endpoints),
**Peripherals** (hardware interfaces), and a **Processor** with its **Architecture**.

The hierarchy is: `Operation → Device → Service → Vulnerability`.

In [ ]:
op.addDevice("web-server-01", "10.0.1.10", operatingsystem="Ubuntu 22.04")
op.addDevice("db-server-01", "10.0.1.20", operatingsystem="Debian 12")

dev = op.getDeviceByHostname("web-server-01")
dev.addService(portNumber=443, app="nginx", protocol="ipv4", transport_layer="HTTPS")
dev.addService(portNumber=22, app="openssh", protocol="ipv4", transport_layer="SSH")

print(f"Devices: {[d.hostname for d in op.devices]}")
print(f"Services on {dev.hostname}: {[(s.portNumber, s.app) for s in dev.services]}")

## Vulnerabilities

**Vulnerabilities** attach to services (or devices, cloud accounts). Each vulnerability
has a CVSS score, risk rating, and optional reproduction steps for retesting.

In [ ]:
svc = dev.services[0]  # HTTPS service
svc.addVulnerability(
    title="SQL Injection in /api/search",
    cvss=9,
    description="Union-based SQLi in search parameter",
    threat="critical",
    risk={"likelihood": "High", "impact": "High", "severity": "Critical"},
)
svc.addVulnerability(
    title="Missing HSTS Header",
    cvss=4,
    description="HTTP Strict Transport Security not configured",
    threat="medium",
    risk={"likelihood": "Medium", "impact": "Low", "severity": "Low"},
)

print(f"Vulnerabilities on port {svc.portNumber}:")
for v in svc.vulnerabilities:
    print(f"  [{v.cvss}] {v.title}")

## Cloud Accounts

Wintermute models cloud infrastructure via **CloudAccount** subclasses. The **AWSAccount**
type supports IAM users, IAM roles, services, and per-account vulnerabilities.

In [ ]:
op.addAWSAccount("acme-prod", account_id="123456789012")
aws = op.awsaccounts[0]

print(f"Cloud Account: {aws.name} (ID: {aws.account_id})")
print(f"Cloud type: {aws.cloud_type}")
print(f"Provider: {aws.provider}")

## Serialization

Every Wintermute model inherits `to_dict()` and `from_dict()` from **BaseModel**.
This handles nested objects, enums, datetimes (ISO 8601), and IP addresses automatically.

In [ ]:
import json

data = op.to_dict()
print(json.dumps(data, indent=2, default=str)[:2000])

## JSON File Backend

The **JsonFileBackend** persists operations as JSON files on disk. Wintermute uses a
protocol-based backend system — register a backend, then save/load by operation name.

In [ ]:
import os
import tempfile

from wintermute.backends.json_storage import JsonFileBackend

with tempfile.TemporaryDirectory() as tmpdir:
    backend = JsonFileBackend(base_path=tmpdir)
    Operation.register_backend("json", backend, make_default=True)

    # Save the operation
    data = op.to_dict()
    result = backend.save(op.operation_name, data)
    print(f"Saved: {result}")
    print(f"Files: {os.listdir(tmpdir)}")

    # Load it back
    loaded_data = backend.load(op.operation_name)
    loaded_op = Operation.from_dict(loaded_data)
    print(f"Loaded operation: {loaded_op.operation_name}")
    print(f"Devices: {[d.hostname for d in loaded_op.devices]}")
    print(
        f"Vulns: {[v.title for v in loaded_op.devices[0].services[0].vulnerabilities]}"
    )